## 1. Importación de librerías y conexión a MySQL
Importamos pandas, pymysql y sqlalchemy, y creamos la conexión
a nuestra instancia local de MySQL.

In [1]:
import pandas as pd
import pymysql
from sqlalchemy import create_engine, text
import getpass

usuario = 'root'
contraseña = getpass.getpass('Introduce tu contraseña de MySQL: ')
host = 'localhost'
puerto = '3306'

engine = create_engine(f'mysql+pymysql://{usuario}:{contraseña}@{host}:{puerto}')

print('Conexión creada')

Conexión creada


**Conclusiones:**
- Conexión a MySQL establecida correctamente desde Python usando SQLAlchemy + pymysql
- La contraseña se introdujo de forma segura con getpass, sin quedar escrita en el notebook
- El siguiente paso es crear una base de datos para el proyecto y cargar en ella el dataset limpio

## 2. Creación de la base de datos del proyecto
Creamos una base de datos nueva en MySQL llamada "dashboard1_pantallas"
para almacenar los datos del Dashboard 1.

In [2]:
with engine.connect() as conn:
    conn.execute(text("CREATE DATABASE IF NOT EXISTS dashboard1_pantallas"))

print('Base de datos creada (o ya existía)')

Base de datos creada (o ya existía)


**Conclusiones:**
- Base de datos `dashboard1_pantallas` creada correctamente en MySQL
- Igual que hizo tu compañera con `dashboard2_pantallas`, esta base de datos
  es visible también desde MySQL Workbench (misma instancia, mismo servidor)
- El IF NOT EXISTS evita errores si se vuelve a ejecutar la celda

## 3. Carga del CSV limpio en una tabla de MySQL
Leemos dataset_limpio_dashboard1.csv y lo cargamos como una tabla
llamada "contenido_infantil" dentro de la base de datos dashboard1_pantallas.

In [3]:
df = pd.read_csv('dataset_limpio_dashboard1.csv')

engine_db = create_engine(f'mysql+pymysql://{usuario}:{contraseña}@{host}:{puerto}/dashboard1_pantallas')

df.to_sql('contenido_infantil', engine_db, if_exists='replace', index=False)

print('Tabla contenido_infantil creada con', len(df), 'filas')

Tabla contenido_infantil creada con 3353 filas


**Conclusiones:**
- Tabla `contenido_infantil` creada correctamente en `dashboard1_pantallas` con 3.353 filas
- Las 11 columnas del CSV limpio se han cargado correctamente en MySQL
- La tabla ya puede consultarse con SQL tanto desde Python como desde MySQL Workbench

## 4. Primera consulta: vista general de la tabla
Comprobamos que la tabla se cargó correctamente consultando
las primeras filas directamente con SQL.

In [4]:
query = "SELECT * FROM contenido_infantil LIMIT 5"
pd.read_sql(query, engine_db)

,type,title,country,date_added,release_year,rating,duration,listed_in,anio_agregado,plataforma,clasificacion_edad
0,Movie,My Little Pony: A New Generation,None,"September 24, 2021",2021,PG,91 min,Children & Family Movies,2021.0,Netflix,Familiar (supervisión recomendada)
1,Movie,Confessions of an Invisible Girl,None,"September 22, 2021",2021,TV-PG,91 min,"Children & Family Movies, Comedies",2021.0,Netflix,Familiar (supervisión recomendada)
2,Movie,Avvai Shanmughi,None,"September 21, 2021",1996,TV-PG,161 min,"Comedies, International Movies",2021.0,Netflix,Familiar (supervisión recomendada)
3,Movie,Go! Go! Cory Carson: Chrissy Takes the Wheel,None,"September 21, 2021",2021,TV-Y,61 min,Children & Family Movies,2021.0,Netflix,Todos los públicos (0+)
4,Movie,Minsara Kanavu,None,"September 21, 2021",1997,TV-PG,147 min,"Comedies, International Movies, Music & Musicals",2021.0,Netflix,Familiar (supervisión recomendada)


**Conclusiones:**
- La consulta SQL funciona correctamente desde Python
- Los datos coinciden con el CSV limpio: 11 columnas, valores correctos
- Los nulos de country aparecen como None (representación de NULL en MySQL)
- Las columnas nuevas creadas en el EDA (anio_agregado, plataforma,
  clasificacion_edad) se cargaron correctamente
- La tabla está lista para hacer consultas más complejas

## 5. Consulta agregada: títulos infantiles por plataforma
Calculamos cuántos títulos infantiles tiene cada plataforma
y qué porcentaje representa sobre el total.

In [5]:
query = """
SELECT 
    plataforma,
    COUNT(*) AS num_titulos,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM contenido_infantil), 2) AS porcentaje_total
FROM contenido_infantil
GROUP BY plataforma
ORDER BY num_titulos DESC
"""

pd.read_sql(query, engine_db)

,plataforma,num_titulos,porcentaje_total
0,Netflix,2054,61.26
1,Disney+,1299,38.74


**Conclusiones:**
- Netflix tiene 2.054 títulos infantiles (61.26% del total)
- Disney+ tiene 1.299 títulos infantiles (38.74% del total)
- Aunque Netflix tiene más títulos infantiles en números absolutos,
  hay que recordar que esto representa solo el 23.4% de su catálogo total,
  mientras que Disney+ dedica el 89.9% de su catálogo al público infantil
- Esta consulta confirma con SQL los resultados vistos en el paso 15 del EDA

## 6. Consulta: evolución del contenido infantil por año y plataforma
Calculamos cuántos títulos infantiles se añadieron cada año
en cada plataforma, para ver la tendencia temporal.

In [6]:
query = """
SELECT 
    plataforma,
    anio_agregado,
    COUNT(*) AS num_titulos
FROM contenido_infantil
WHERE anio_agregado >= 2015
GROUP BY plataforma, anio_agregado
ORDER BY plataforma, anio_agregado
"""

pd.read_sql(query, engine_db)

,plataforma,anio_agregado,num_titulos
0,Disney+,2019.0,724
1,Disney+,2020.0,316
2,Disney+,2021.0,259
3,Netflix,2015.0,25
4,Netflix,2016.0,116
5,Netflix,2017.0,282
6,Netflix,2018.0,338
7,Netflix,2019.0,428
8,Netflix,2020.0,459
9,Netflix,2021.0,347


**Conclusiones:**
- Los datos confirman con SQL la tendencia vista en el paso 16 del EDA:
    - Netflix: crecimiento progresivo de 25 títulos en 2015 hasta 459 en 2020,
      con una bajada en 2021 (347)
    - Disney+: pico enorme en 2019 (724 títulos) por el lanzamiento de la plataforma,
      seguido de una bajada progresiva en 2020 (316) y 2021 (259)
- Netflix multiplica por 18 su catálogo infantil entre 2015 y 2020
- Esta tabla es perfecta para construir el gráfico de líneas de tendencia
  temporal en Tableau

## 7. Consulta: distribución por tipo de contenido y plataforma
Calculamos cuántas películas y series tiene cada plataforma
en su catálogo infantil.

In [7]:
query = """
SELECT 
    plataforma,
    type AS tipo,
    COUNT(*) AS num_titulos,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY plataforma), 2) AS porcentaje
FROM contenido_infantil
GROUP BY plataforma, type
ORDER BY plataforma, num_titulos DESC
"""

pd.read_sql(query, engine_db)

,plataforma,tipo,num_titulos,porcentaje
0,Disney+,Movie,948,72.98
1,Disney+,TV Show,351,27.02
2,Netflix,Movie,1269,61.78
3,Netflix,TV Show,785,38.22


**Conclusiones:**
- Confirmación con SQL de los resultados del paso 17 del EDA:
    - Disney+: 72.98% películas vs 27.02% series
    - Netflix: 61.78% películas vs 38.22% series
- En ambas plataformas dominan las películas sobre las series
- Disney+ tiene una proporción aún mayor de películas que Netflix,
  coherente con su enorme archivo histórico de largometrajes animados
- Se usó una window function (OVER PARTITION BY plataforma) para calcular
  el porcentaje dentro de cada plataforma en una sola consulta.

## 8. Consulta: distribución por clasificación de edad y plataforma
Analizamos qué clasificación de edad predomina en cada plataforma
para el contenido infantil.

In [8]:
query = """
SELECT 
    plataforma,
    clasificacion_edad,
    COUNT(*) AS num_titulos,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY plataforma), 2) AS porcentaje
FROM contenido_infantil
GROUP BY plataforma, clasificacion_edad
ORDER BY plataforma, num_titulos DESC
"""

pd.read_sql(query, engine_db)

,plataforma,clasificacion_edad,num_titulos,porcentaje
0,Disney+,Familiar (supervisión recomendada),855,65.82
1,Disney+,Todos los públicos (0+),302,23.25
2,Disney+,Mayores de 7 años,142,10.93
3,Netflix,Familiar (supervisión recomendada),1368,66.60
4,Netflix,Todos los públicos (0+),347,16.89
5,Netflix,Mayores de 7 años,339,16.50


**Conclusiones:**
- En ambas plataformas el contenido "Familiar (supervisión recomendada)" domina
  con ~66% en las dos (Disney+ 65.82%, Netflix 66.60%)
- La diferencia clave está en las otras dos categorías:
    - Disney+ tiene más contenido "Todos los públicos (0+)": 23.25% vs 16.89% de Netflix
    - Netflix tiene más contenido "Mayores de 7 años": 16.50% vs 10.93% de Disney+
- Esto confirma que Disney+ es proporcionalmente más seguro para los más pequeños,
  con casi 1 de cada 4 títulos apto desde 0 años

## 9. Consulta: antigüedad media del contenido al incorporarse a la plataforma
Calculamos cuántos años de media tiene un título cuando se añade
a cada plataforma, para ver si apuestan por contenido nuevo o clásico.

In [9]:
query = """
SELECT 
    plataforma,
    ROUND(AVG(anio_agregado - release_year), 2) AS antiguedad_media,
    MIN(release_year) AS titulo_mas_antiguo,
    MAX(release_year) AS titulo_mas_reciente,
    COUNT(*) AS num_titulos
FROM contenido_infantil
WHERE anio_agregado IS NOT NULL
GROUP BY plataforma
ORDER BY antiguedad_media DESC
"""

pd.read_sql(query, engine_db)

,plataforma,antiguedad_media,titulo_mas_antiguo,titulo_mas_reciente,num_titulos
0,Disney+,18.03,1928,2021,1299
1,Netflix,5.19,1943,2021,2013


**Conclusiones:**
- Disney+ incorpora contenido con una antigüedad media de 18 años,
  frente a solo 5 años de Netflix
- Esto confirma la naturaleza diferente de cada plataforma:
    - Disney+ es un archivo histórico: su título más antiguo data de 1928
      (los primeros cortometrajes de Mickey Mouse) y la media de 18 años
      refleja décadas de clásicos animados
    - Netflix apuesta por contenido reciente: antigüedad media de solo 5 años,
      lo que significa que la mayoría de su catálogo infantil se produce
      y añade casi al mismo tiempo
- Para el Dashboard 1: este dato resume perfectamente la diferencia
  de estrategia entre ambas plataformas en una sola cifra

## 10. Consulta: agrupación por décadas de estreno
Clasificamos el contenido infantil por décadas para ver
en qué épocas se produjo más contenido que hoy consumimos.

In [10]:
query = """
SELECT 
    plataforma,
    CASE
        WHEN release_year < 1980 THEN 'Antes de 1980'
        WHEN release_year BETWEEN 1980 AND 1989 THEN 'Años 80'
        WHEN release_year BETWEEN 1990 AND 1999 THEN 'Años 90'
        WHEN release_year BETWEEN 2000 AND 2009 THEN 'Años 2000'
        WHEN release_year BETWEEN 2010 AND 2019 THEN 'Años 2010'
        ELSE 'Años 2020'
    END AS decada,
    COUNT(*) AS num_titulos
FROM contenido_infantil
GROUP BY plataforma, decada
ORDER BY plataforma, decada
"""

pd.read_sql(query, engine_db)

,plataforma,decada,num_titulos
0,Disney+,Años 2000,261
1,Disney+,Años 2010,483
2,Disney+,Años 2020,180
3,Disney+,Años 80,49
4,Disney+,Años 90,133
5,Disney+,Antes de 1980,193
6,Netflix,Años 2000,192
7,Netflix,Años 2010,1345
8,Netflix,Años 2020,377
9,Netflix,Años 80,33


**Conclusiones:**
- Netflix concentra el 85% de su catálogo infantil en los años 2010 y 2020
  (1.345 + 377 = 1.722 títulos), confirmando su apuesta por contenido reciente
- Disney+ tiene una distribución mucho más equilibrada entre décadas:
    - Antes de 1980: 193 títulos (clásicos históricos)
    - Años 90: 133 títulos (época dorada Disney)
    - Años 2000: 261 títulos
    - Años 2010: 483 títulos (su década más productiva)
- Disney+ tiene 193 títulos anteriores a 1980, frente a solo 49 de Netflix:
  casi 4 veces más contenido clásico
- Para el Dashboard 1: esta tabla es perfecta para un gráfico de barras
  agrupadas por décadas en Tableau, mostrando visualmente la diferencia
  de estrategia entre ambas plataformas

## 11. Resumen estadístico general en SQL
Calculamos min, max, media y desviación estándar de las variables
numéricas principales, equivalente al describe() de pandas.

In [11]:
query = """
SELECT 
    'release_year' AS variable,
    MIN(release_year) AS minimo,
    MAX(release_year) AS maximo,
    ROUND(AVG(release_year), 2) AS media,
    ROUND(STDDEV(release_year), 2) AS desviacion
FROM contenido_infantil

UNION ALL

SELECT 
    'anio_agregado',
    MIN(anio_agregado),
    MAX(anio_agregado),
    ROUND(AVG(anio_agregado), 2),
    ROUND(STDDEV(anio_agregado), 2)
FROM contenido_infantil
WHERE anio_agregado IS NOT NULL
"""

pd.read_sql(query, engine_db)

,variable,minimo,maximo,media,desviacion
0,release_year,1928.0,2021.0,2008.97,17.07
1,anio_agregado,2011.0,2021.0,2019.16,1.43


**Conclusiones:**
- Los resultados son consistentes con las estadísticas descriptivas del EDA,
  confirmando que la carga a MySQL no alteró los datos
- release_year: rango 1928-2021, media 2008.97, desviación 17.07
  → gran dispersión por el archivo histórico de Disney+
- anio_agregado: rango 2011-2021, media 2019.16, desviación muy baja (1.43)
  → la mayoría del contenido se añadió en un período muy concentrado (2019-2021)
- El UNION ALL permite obtener estadísticas de varias columnas en una
  sola consulta sin necesidad de hacer varias llamadas separadas

## 12. Creación de una VIEW resumen para Tableau
Creamos una vista en MySQL con columnas renombradas en español,
lista para conectar directamente desde Tableau.

In [12]:
query_view = """
CREATE OR REPLACE VIEW vista_dashboard1 AS
SELECT 
    type AS tipo,
    title AS titulo,
    country AS pais,
    release_year AS anio_estreno,
    anio_agregado,
    rating,
    clasificacion_edad,
    listed_in AS generos,
    duration AS duracion,
    plataforma
FROM contenido_infantil
"""

with engine_db.connect() as conn:
    conn.execute(text(query_view))

print('Vista vista_dashboard1 creada correctamente')

Vista vista_dashboard1 creada correctamente


**Conclusiones:**
- Vista `vista_dashboard1` creada correctamente en `dashboard1_pantallas`
- Renombra las columnas técnicas en inglés a nombres claros en español:
  type→tipo, title→titulo, country→pais, release_year→anio_estreno,
  listed_in→generos, duration→duracion
- Esta vista es la que se conectará desde Tableau para construir el Dashboard 1,
  evitando tener que renombrar columnas dentro de Tableau

## 📌 Conclusiones generales del SQL — Dashboard 1

### Sobre la base de datos
- Base de datos `dashboard1_pantallas` creada en MySQL con una tabla
  principal `contenido_infantil` (3.353 filas, 11 columnas) y una
  vista `vista_dashboard1` lista para Tableau

### Consultas realizadas
- Títulos infantiles por plataforma: Netflix 61.26% vs Disney+ 38.74%
- Evolución temporal: Netflix creció x18 entre 2015 y 2020;
  Disney+ cargó 724 títulos en su lanzamiento de 2019
- Tipo de contenido: ambas plataformas dominadas por películas
  (Netflix 61.78%, Disney+ 72.98%)
- Clasificación de edad: ~66% del contenido es "Familiar" en ambas,
  Disney+ tiene más contenido apto desde 0 años (23.25% vs 16.89%)
- Antigüedad media: Disney+ incorpora contenido con 18 años de media
  frente a 5 años de Netflix
- Por décadas: Netflix concentra el 85% en 2010-2020;
  Disney+ tiene distribución histórica desde 1928

### Técnicas SQL utilizadas
- GROUP BY y COUNT para agregaciones
- Window functions (OVER PARTITION BY) para porcentajes dentro de grupo
- CASE WHEN para agrupar por décadas
- UNION ALL para resumen estadístico multi-variable
- CREATE VIEW para preparar los datos para Tableau

### Producto final
- Vista `vista_dashboard1` disponible en MySQL lista para conectar con Tableau